# 2. Scanpy QC and Scrublet Doublet Detection

This notebook performs quality control filtering and doublet detection on CellBender-corrected data.

**Input:** CellBender-corrected h5 files (`*_cellbender_output_filtered.h5`)

**Output:** QC-filtered h5ad files with doublet annotations (`*_qc_scrub.h5ad`)

## QC Filtering Criteria
- `log10GenesPerUMI > 0.8` (complexity filter)
- `min_counts = 500` (minimum UMIs per cell)
- `min_genes = 250` (minimum genes per cell)
- `pct_counts_mt < 20` (maximum mitochondrial percentage)
- `min_cells = 10` (minimum cells per gene)

## Scrublet Parameters
- `sim_doublet_ratio = 2`
- `expected_doublet_rate = 0.10`

## Setup

In [ ]:
import scanpy as sc
import math
import pandas as pd
import os

print(f"Scanpy version: {sc.__version__}")

## Define Paths and Samples

In [ ]:
# Set working directory - update this path to your data location
data_dir = "./"  # Directory containing CellBender output files
output_dir = "./"  # Directory for QC'd h5ad output

# Sample names
samples = [
    "CCM2WTWeek0",
    "CCM2HetWeek0",
    "CCM2WTWeek12",
    "CCM2HetWeek12"
]

## Load CellBender Output

Load the filtered CellBender output files.

In [ ]:
adata_dict = {}

for sample in samples:
    print(f"Loading {sample}...")
    filepath = os.path.join(data_dir, f"{sample}_cellbender_output_filtered.h5")
    adata = sc.read_10x_h5(filepath)
    adata.var_names_make_unique()
    adata_dict[sample] = adata
    print(f"  Loaded {adata.n_obs} cells, {adata.n_vars} genes")

## QC and Scrublet Pipeline

For each sample:
1. Annotate gene types (mitochondrial, ribosomal, hemoglobin)
2. Calculate QC metrics
3. Filter cells and genes
4. Run Scrublet for doublet detection
5. Save filtered h5ad file

In [ ]:
def run_qc_scrublet(adata, sample_name, output_dir):
    """
    Run QC filtering and Scrublet doublet detection.
    
    Parameters
    ----------
    adata : AnnData
        Input AnnData object
    sample_name : str
        Sample identifier
    output_dir : str
        Output directory for h5ad file
        
    Returns
    -------
    AnnData
        Filtered AnnData with doublet annotations
    """
    print(f"\n{'='*60}")
    print(f"Processing: {sample_name}")
    print(f"{'='*60}")
    
    # Starting counts
    print(f"Starting cells: {adata.n_obs}")
    print(f"Starting genes: {adata.n_vars}")
    
    # Annotate gene types
    adata.var["mt"] = adata.var_names.str.startswith("mt-")
    adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
    adata.var["hb"] = adata.var_names.str.contains("^Hb[^(p)]")
    
    # Calculate QC metrics
    sc.pp.calculate_qc_metrics(
        adata, 
        qc_vars=["mt", "ribo", "hb"], 
        inplace=True, 
        log1p=True
    )
    
    # Calculate complexity (log10 genes per UMI)
    adata.obs["log10genesperUMI"] = (
        adata.obs["n_genes_by_counts"].apply(math.log10) / 
        adata.obs["total_counts"].apply(math.log10)
    )
    
    # Apply filters
    print("\nApplying QC filters...")
    
    # Complexity filter
    adata = adata[adata.obs["log10genesperUMI"] > 0.8].copy()
    print(f"  After complexity filter (log10GenesPerUMI > 0.8): {adata.n_obs} cells")
    
    # Minimum counts filter
    sc.pp.filter_cells(adata, min_counts=500)
    print(f"  After min_counts=500: {adata.n_obs} cells")
    
    # Minimum genes filter
    sc.pp.filter_cells(adata, min_genes=250)
    print(f"  After min_genes=250: {adata.n_obs} cells")
    
    # Mitochondrial filter
    adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
    print(f"  After pct_counts_mt < 20: {adata.n_obs} cells")
    
    # Gene filter
    sc.pp.filter_genes(adata, min_cells=10)
    print(f"  After min_cells=10: {adata.n_vars} genes")
    
    # Run Scrublet
    print("\nRunning Scrublet for doublet detection...")
    adata = sc.pp.scrublet(
        adata,
        sim_doublet_ratio=2,
        expected_doublet_rate=0.10,
        copy=True
    )
    
    # Report doublet statistics
    n_doublets = adata.obs["predicted_doublet"].sum()
    pct_doublets = 100 * n_doublets / adata.n_obs
    print(f"  Detected doublets: {n_doublets} ({pct_doublets:.1f}%)")
    
    # Ensure unique variable names
    adata.var_names_make_unique()
    
    # Save output
    output_path = os.path.join(output_dir, f"{sample_name}_qc_scrub.h5ad")
    adata.write_h5ad(filename=output_path)
    print(f"\nSaved to: {output_path}")
    print(f"Final: {adata.n_obs} cells, {adata.n_vars} genes")
    
    return adata

In [ ]:
# Process all samples
processed_dict = {}

for sample in samples:
    # Reload fresh copy from CellBender output
    filepath = os.path.join(data_dir, f"{sample}_cellbender_output_filtered.h5")
    adata = sc.read_10x_h5(filepath)
    adata.var_names_make_unique()
    
    # Run QC and Scrublet
    processed = run_qc_scrublet(adata, sample, output_dir)
    processed_dict[sample] = processed

## Summary

Print summary of all processed samples.

In [ ]:
print("\n" + "="*60)
print("PROCESSING SUMMARY")
print("="*60)

summary_data = []
for sample, adata in processed_dict.items():
    n_doublets = adata.obs["predicted_doublet"].sum()
    summary_data.append({
        "Sample": sample,
        "Cells": adata.n_obs,
        "Genes": adata.n_vars,
        "Doublets": n_doublets,
        "Doublet %": f"{100*n_doublets/adata.n_obs:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

## Output Files

The following h5ad files are ready for conversion to Seurat format:
- `CCM2WTWeek0_qc_scrub.h5ad`
- `CCM2HetWeek0_qc_scrub.h5ad`
- `CCM2WTWeek12_qc_scrub.h5ad`
- `CCM2HetWeek12_qc_scrub.h5ad`

Next step: Run `03_h5ad_to_seurat.R` to convert to Seurat RDS format.